# Inject into Pinecone



In [ ]:
import pickle
import time
from pathlib import Path

from dotenv import load_dotenv
from pinecone import Pinecone

load_dotenv(dotenv_path=Path("../.env"))

INPUT_FILE = Path("./embeddings.pkl")
PINECONE_INDEX_NAME = "sqldatabasewithmcp"
UPSERT_BATCH_SIZE = 100
NAMESPACE = ""

In [2]:
with open(INPUT_FILE, "rb") as f:
    data = pickle.load(f)

child_chunks = data["child_chunks"]
parent_chunks = data["parent_chunks"]

parent_lookup = {p["chunk_id"]: p["text"] for p in parent_chunks}

print(f"Loaded {len(child_chunks)} child chunks with embeddings")
print(f"Loaded {len(parent_chunks)} parent chunks for context")
print(f"Vector dimension: {len(child_chunks[0]['embedding'])}")

Loaded 649 child chunks with embeddings
Loaded 176 parent chunks for context
Vector dimension: 1024


In [3]:
pc = Pinecone()
index = pc.Index(PINECONE_INDEX_NAME)

stats = index.describe_index_stats()
print(f"Connected to index: {PINECONE_INDEX_NAME}")
print(f"Dimension: {stats.dimension}")
print(f"Current vector count: {stats.total_vector_count}")

c:\project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to index: sqldatabasewithmcp
Dimension: 1024
Current vector count: 82


In [4]:
vectors = []

for chunk in child_chunks:
    metadata = {
        "text_chunk": chunk["text"],
        "doc_id": chunk["doc_id"],
        "filename": chunk["filename"],
        "parent_id": chunk.get("parent_id", ""),
        "chunk_index": chunk["chunk_index"],
        "token_count": chunk["token_count"],
        "is_child": True,
    }

    vectors.append({
        "id": chunk["chunk_id"],
        "values": chunk["embedding"],
        "metadata": metadata,
    })

print(f"Prepared {len(vectors)} vectors for upsert")

Prepared 649 vectors for upsert


In [5]:
total_batches = (len(vectors) + UPSERT_BATCH_SIZE - 1) // UPSERT_BATCH_SIZE
upserted = 0
start = time.time()

for i in range(0, len(vectors), UPSERT_BATCH_SIZE):
    batch = vectors[i : i + UPSERT_BATCH_SIZE]
    batch_num = i // UPSERT_BATCH_SIZE + 1

    index.upsert(vectors=batch, namespace=NAMESPACE)
    upserted += len(batch)

    print(f"Batch {batch_num}/{total_batches} — {len(batch)} vectors upserted ({upserted}/{len(vectors)})")

    if batch_num < total_batches:
        time.sleep(0.5)

elapsed = time.time() - start
print(f"\nUpserted {upserted} vectors in {elapsed:.1f}s")

Batch 1/7 — 100 vectors upserted (100/649)
Batch 2/7 — 100 vectors upserted (200/649)
Batch 3/7 — 100 vectors upserted (300/649)
Batch 4/7 — 100 vectors upserted (400/649)
Batch 5/7 — 100 vectors upserted (500/649)
Batch 6/7 — 100 vectors upserted (600/649)
Batch 7/7 — 49 vectors upserted (649/649)

Upserted 649 vectors in 22.0s


In [6]:
time.sleep(10)

stats = index.describe_index_stats()
print(f"Index: {PINECONE_INDEX_NAME}")
print(f"Total vectors: {stats.total_vector_count}")
print(f"Dimension: {stats.dimension}")

if stats.namespaces:
    for ns, ns_stats in stats.namespaces.items():
        label = ns if ns else "(default)"
        print(f"  Namespace '{label}': {ns_stats.vector_count} vectors")

print(f"\nInjection complete!")

Index: sqldatabasewithmcp
Total vectors: 731
Dimension: 1024
  Namespace '(default)': 731 vectors

Injection complete!
